# 多层感知机：数学推导

这一篇只关注公式本身：线性层怎么求导，ReLU 的反向传播怎么写，为什么 $\frac{\partial L}{\partial W}=g h^T$，以及 Jacobian 在这里到底表示什么。

数学推导默认采用列向量约定。对单个样本：

$$
h=\operatorname{ReLU}(W_1x+b_1)
$$

$$
\hat y=W_2h+b_2
$$

$$
L=\frac{1}{2}(\hat y-y)^2
$$


## 反向传播

从损失函数开始往回推：

$$
\frac{\partial L}{\partial \hat y}=\hat y-y
$$

反向传播时，每个梯度的尺寸和它对应的原变量一致：

| 梯度 | 对应变量尺寸 | 本例尺寸 |
|---|---:|---:|
| $\frac{\partial L}{\partial \hat y}$ | $J\times 1$ | $1\times 1$ |
| $\frac{\partial L}{\partial W_2}$ | $J\times I$ | $1\times 2$ |
| $\frac{\partial L}{\partial b_2}$ | $J\times 1$ | $1\times 1$ |
| $\frac{\partial L}{\partial h}$ | $I\times 1$ | $2\times 1$ |
| $\frac{\partial L}{\partial z_1}$ | $I\times 1$ | $2\times 1$ |
| $\frac{\partial L}{\partial W_1}$ | $I\times D$ | $2\times 1$ |
| $\frac{\partial L}{\partial b_1}$ | $I\times 1$ | $2\times 1$ |

输出层在列向量约定下是:

$$
\hat y = W_2 h + b_2
$$

所以输出层梯度是：

$$
\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial \hat y}h^T
$$

$$
\frac{\partial L}{\partial b_2}=\frac{\partial L}{\partial \hat y}
$$

$$
\frac{\partial L}{\partial h}=W_2^T\frac{\partial L}{\partial \hat y}
$$

### Jacobian 是什么

如果一个函数的输入和输出都是标量：

$$
y=f(x)
$$

那么导数 $\frac{dy}{dx}$ 是一个数。

但神经网络里经常遇到“向量到向量”的函数：

$$
y=f(x),\quad x\in\mathbb{R}^{n\times 1},\quad y\in\mathbb{R}^{m\times 1}
$$

这时每个输出 $y_i$ 都可能依赖每个输入 $x_j$，所以会有 $m\times n$ 个偏导数。把它们排成矩阵，就是 Jacobian：

$$
J=
\begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \frac{\partial y_1}{\partial x_2} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\frac{\partial y_2}{\partial x_1} & \frac{\partial y_2}{\partial x_2} & \cdots & \frac{\partial y_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \frac{\partial y_m}{\partial x_2} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}
$$

也就是：

$$
J_{i,j}=\frac{\partial y_i}{\partial x_j}
$$

例如在线性层：

$$
y=Wh+b
$$

其中：

$$
h\in\mathbb{R}^{I\times 1},\quad
W\in\mathbb{R}^{J\times I},\quad
y\in\mathbb{R}^{J\times 1}
$$

如果对输入向量 $h$ 求导，$y$ 有 $J$ 个分量，$h$ 有 $I$ 个分量，所以 $\frac{\partial y}{\partial h}$ 是一个普通的 Jacobian 矩阵：

$$
\frac{\partial y}{\partial h}
=
\begin{bmatrix}
\frac{\partial y_1}{\partial h_1} & \frac{\partial y_1}{\partial h_2} & \cdots & \frac{\partial y_1}{\partial h_I} \\
\frac{\partial y_2}{\partial h_1} & \frac{\partial y_2}{\partial h_2} & \cdots & \frac{\partial y_2}{\partial h_I} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial y_J}{\partial h_1} & \frac{\partial y_J}{\partial h_2} & \cdots & \frac{\partial y_J}{\partial h_I}
\end{bmatrix}
\in\mathbb{R}^{J\times I}
$$

因为 $y_j=\sum_i W_{j,i}h_i+b_j$，所以上面这个 Jacobian 实际就是：

$$
\frac{\partial y}{\partial h}=W
$$

更复杂的是 $\frac{\partial y}{\partial W}$。因为 $y$ 是向量，$W$ 是矩阵，问“$y$ 对 $W$ 的导数”时，就要同时说明：

- 是哪个输出分量 $y_j$
- 是哪个权重元素 $W_{k,i}$

所以它的基本元素不是 $\frac{\partial y_j}{\partial h_i}$ 这种两个下标，而是：

$$
\frac{\partial y_j}{\partial W_{k,i}}
$$

这里有三个自由下标：$j,k,i$。因此 $\frac{\partial y}{\partial W}$ 不能自然排成普通二维矩阵，而是可以理解成一个三维的导数对象：

$$
\left[\frac{\partial y_j}{\partial W_{k,i}}\right]
\quad \text{形状可理解为 } J\times J\times I
$$

这里两个 $J$ 的含义不同：

- 第一个 $J$：输出分量下标 $j$，表示正在看哪个 $y_j$
- 第二个 $J$：权重矩阵的行下标 $k$，表示正在看哪一行权重 $W_{k,i}$
- $I$：权重矩阵的列下标 $i$，表示正在看哪一个输入/隐藏层分量

这两个 $J$ 的含义不同，但取值范围相同，都是 $1,2,\cdots,J$。原因是输出向量 $y$ 有 $J$ 个分量，而权重矩阵 $W\in\mathbb{R}^{J\times I}$ 也有 $J$ 行；第 $k$ 行权重专门用来计算第 $k$ 个输出。

可以把几个典型位置展开看一下。先写出第 $j$ 个输出：

$$
y_j=\sum_i W_{j,i}h_i+b_j
$$

如果看第一个输出，也就是固定 $j=1$：

$$
y_1=\sum_i W_{1,i}h_i+b_1
$$

注意这里固定的是输出下标 $j=1$，但分母里的权重行下标 $k$ 仍然可以取 $1,2,\cdots,J$。也就是说，我们是在问：任意一行权重 $W_{k,i}$ 对第一个输出 $y_1$ 有没有影响？

只有第一行权重 $W_{1,i}$ 出现在 $y_1$ 里面，所以：

$$
\frac{\partial y_1}{\partial W_{1,i}}=h_i
$$

而其他行权重，例如 $W_{2,i},W_{3,i},\cdots,W_{J,i}$，都没有出现在 $y_1$ 的表达式里，所以：

$$
\frac{\partial y_1}{\partial W_{k,i}}=0\quad (k\ne 1)
$$

合起来才写成：

$$
\frac{\partial y_1}{\partial W_{k,i}}
=
\begin{cases}
h_i, & k=1 \\
0, & k\ne 1
\end{cases}
$$

如果看第 $k$ 个输出，也就是让输出下标刚好和权重行下标相同，$j=k$：

$$
y_k=\sum_i W_{k,i}h_i+b_k
$$

这时 $W_{k,i}$ 确实参与了 $y_k$ 的计算，所以：

$$
\frac{\partial y_k}{\partial W_{k,i}}=h_i
$$

如果看最后一个输出，也就是 $j=J$：

$$
y_J=\sum_i W_{J,i}h_i+b_J
$$

这时只有最后一行权重 $W_{J,i}$ 出现在 $y_J$ 里面，所以：

$$
\frac{\partial y_J}{\partial W_{k,i}}
=
\begin{cases}
h_i, & k=J \\
0, & k\ne J
\end{cases}
$$

所以也可以读成：

$$
\text{形状}=\text{输出下标}\times\text{权重行下标}\times\text{权重列下标}
=J\times J\times I
$$

逐元素看，$W_{k,i}$ 只会影响第 $k$ 个输出 $y_k$，不会影响其他输出，所以：

$$
\frac{\partial y_j}{\partial W_{k,i}}
=
\begin{cases}
h_i, & j=k \\
0, & j\ne k
\end{cases}
$$

这就是为什么说 $\frac{\partial y}{\partial W}$ 比普通 Jacobian 更复杂：它不是一个方便直接写下来的二维矩阵。

但是反向传播最终想更新的是参数 $W$，所以我们真正需要的是损失函数对每个参数元素的梯度。它会排成一个和 $W$ 同形状的矩阵：

$$
\frac{\partial L}{\partial W}
=
\begin{bmatrix}
\frac{\partial L}{\partial W_{1,1}} & \frac{\partial L}{\partial W_{1,2}} & \cdots & \frac{\partial L}{\partial W_{1,I}} \\
\frac{\partial L}{\partial W_{2,1}} & \frac{\partial L}{\partial W_{2,2}} & \cdots & \frac{\partial L}{\partial W_{2,I}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial L}{\partial W_{J,1}} & \frac{\partial L}{\partial W_{J,2}} & \cdots & \frac{\partial L}{\partial W_{J,I}}
\end{bmatrix}
\in \mathbb{R}^{J\times I}
$$

也就是说，$\frac{\partial L}{\partial W}$ 的第 $(j,i)$ 个元素，就是“参数 $W_{j,i}$ 变化一点点时，损失 $L$ 会怎么变”：

$$
\left(\frac{\partial L}{\partial W}\right)_{j,i}
=\frac{\partial L}{\partial W_{j,i}}
$$

反向传播通常不会显式构造完整的 $\frac{\partial y}{\partial W}$ Jacobian，而是把上游梯度一起乘进去，直接得到这个和参数同形状的 $\frac{\partial L}{\partial W}$。这就是下面推导 $\frac{\partial L}{\partial W}=g h^T$ 的原因。

### 普适结论：为什么反向传播里是 $\frac{\partial L}{\partial W}=g h^T$

对一般的线性层：

$$
y=Wh+b
$$

其中：

$$
h\in \mathbb{R}^{I\times 1},\quad
W\in \mathbb{R}^{J\times I},\quad
y\in \mathbb{R}^{J\times 1},\quad
b\in \mathbb{R}^{J\times 1}
$$

展开第 $j$ 个输出：

$$
y_j=\sum_i W_{j,i}h_i+b_j
$$

如果直接问 $\frac{\partial y}{\partial W}$，严格来说它不是一个普通矩阵，因为 $y$ 是向量，$W$ 是矩阵。它对应的是一个三维 Jacobian：

$$
\frac{\partial y_j}{\partial W_{k,i}}
$$

逐元素看：

$$
\frac{\partial y_j}{\partial W_{k,i}}
=
\begin{cases}
h_i, & j=k \\
0, & j\ne k
\end{cases}
$$

所以，不能简单说 $\frac{\partial y}{\partial W}=h^T$。真正常用、也最有操作意义的是：给定上游梯度

$$
g=\frac{\partial L}{\partial y}\in \mathbb{R}^{J\times 1}
$$

求损失 $L$ 对参数 $W$ 的梯度。对某个参数 $W_{j,i}$：

$$
\frac{\partial L}{\partial W_{j,i}}
=\sum_k \frac{\partial L}{\partial y_k}\frac{\partial y_k}{\partial W_{j,i}}
$$

因为只有 $k=j$ 这一项不为 0，所以：

$$
\frac{\partial L}{\partial W_{j,i}}
=\frac{\partial L}{\partial y_j}h_i
=g_jh_i
$$

而矩阵乘法 $gh^T$ 的第 $(j,i)$ 个元素正是：

$$
(gh^T)_{j,i}=g_jh_i
$$

因此得到普适的线性层反向传播公式：

$$
\frac{\partial L}{\partial W}=gh^T
=\frac{\partial L}{\partial y}h^T
$$

同时还有：

$$
\frac{\partial L}{\partial h}=W^Tg
=W^T\frac{\partial L}{\partial y}
$$

$$
\frac{\partial L}{\partial b}=g
=\frac{\partial L}{\partial y}
$$

上面已经得到单个样本的结论。下面再从 batch 的逐元素角度推一遍，主要看清楚为什么多个样本的梯度要累加，以及这个结论如何对应到矩阵乘法。

先把输出层按元素展开。设 batch 里的样本编号是 $n$，隐藏层神经元编号是 $i$，输出编号是 $j$。列向量约定下，输出层权重 $W_2$ 的形状是 $J\times I$，所以：

$$
\hat y_{n,j}=\sum_i W_{2,j,i}h_{n,i}+b_{2,j}
$$

如果把 $j=1,2,\dots,J$ 的所有输出都竖着堆起来，就是：

$$
\begin{bmatrix}
\hat y_{n,1} \\
\hat y_{n,2} \\
\vdots \\
\hat y_{n,J}
\end{bmatrix}
=
\begin{bmatrix}
W_{2,1,1} & W_{2,1,2} & \cdots & W_{2,1,I} \\
W_{2,2,1} & W_{2,2,2} & \cdots & W_{2,2,I} \\
\vdots & \vdots & \ddots & \vdots \\
W_{2,J,1} & W_{2,J,2} & \cdots & W_{2,J,I}
\end{bmatrix}
\begin{bmatrix}
h_{n,1} \\
h_{n,2} \\
\vdots \\
h_{n,I}
\end{bmatrix}
+
\begin{bmatrix}
b_{2,1} \\
b_{2,2} \\
\vdots \\
b_{2,J}
\end{bmatrix}
$$

也就是：

$$
\hat y_n = W_2h_n+b_2
$$

现在固定某一个参数 $W_{2,j,i}$。对某个样本 $n$ 来说，它只出现在第 $j$ 个输出 $\hat y_{n,j}$ 里：

$$
\frac{\partial \hat y_{n,j}}{\partial W_{2,j,i}}=h_{n,i}
$$

如果看的是其他输出 $k\ne j$，那里面用的是 $W_{2,k,i}$，不包含 $W_{2,j,i}$，所以：

$$
\frac{\partial \hat y_{n,k}}{\partial W_{2,j,i}}=0 \quad (k\ne j)
$$

因此对单个样本 $n$，链式法则里真正留下来的只有第 $j$ 个输出这一项：

$$
\frac{\partial L}{\partial W_{2,j,i}}\Bigg|_{\text{来自样本 }n}
=
\frac{\partial L}{\partial \hat y_{n,j}}
\frac{\partial \hat y_{n,j}}{\partial W_{2,j,i}}
=
\frac{\partial L}{\partial \hat y_{n,j}}h_{n,i}
$$

同一个参数 $W_{2,j,i}$ 会被 batch 中所有样本共同使用，所以要把所有样本贡献的梯度加起来：

$$
\frac{\partial L}{\partial W_{2,j,i}}
=\sum_n \frac{\partial L}{\partial \hat y_{n,j}}h_{n,i}
$$

再看矩阵乘法 $\frac{\partial L}{\partial \hat y}h^T$ 的第 $(j,i)$ 个元素：

$$
\left(\frac{\partial L}{\partial \hat y}h^T\right)_{j,i}
=\sum_n \frac{\partial L}{\partial \hat y_{n,j}}h^T_{i,n}
=\sum_n \frac{\partial L}{\partial \hat y_{n,j}}h_{n,i}
$$

这和上面逐元素推出来的式子完全一样，所以：

$$
\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial \hat y}h^T
$$

也就是说，$h^T$ 放在后面，本质上是在沿 batch 维度把“所有样本对同一个参数的梯度贡献”累加起来。矩阵形状匹配只是这个推导的结果，不是出发点。

同理，$b_{2,j}$ 也会被 batch 中所有样本的第 $j$ 个输出共同使用：

$$
\frac{\partial \hat y_{n,j}}{\partial b_{2,j}}=1
$$

所以：

$$
\frac{\partial L}{\partial b_{2,j}}
=\sum_n \frac{\partial L}{\partial \hat y_{n,j}}
$$

如果把 batch 也按列向量约定组织，令 $H=[h_1,h_2,\dots,h_N]$，$\hat Y=[\hat y_1,\hat y_2,\dots,\hat y_N]$，则：

$$
\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial \hat Y}H^T
$$

$$
\frac{\partial L}{\partial b_2}=\sum_n \frac{\partial L}{\partial \hat y_n}
$$

不过 NumPy / PyTorch 代码通常把 batch 放在第 0 维，也就是每个样本是一行。于是代码中的矩阵正好是上面列向量公式的转置版本：

$$
\hat Y_{\text{code}}=H_{\text{code}}W_{2,\text{code}}+b_{2,\text{code}}
$$

对应代码里的梯度写成：

```python
dW2 = h.T @ dy
db2 = dy.sum(axis=0, keepdims=True)
```

隐藏层：

$$
\frac{\partial L}{\partial h}=W_2^T\frac{\partial L}{\partial \hat y}
$$

$$
\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial h}\odot \operatorname{ReLU}'(z_1)
$$

这里的 $\odot$ 表示逐元素乘法，也就是两个同形状向量对应位置相乘。因为 $h=\operatorname{ReLU}(z_1)$ 是对每个隐藏神经元分别做激活，所以反向传播时，每个位置的梯度也只乘自己对应位置的 ReLU 导数：

$$
\frac{\partial L}{\partial z_{1,i}}
=
\frac{\partial L}{\partial h_i}\operatorname{ReLU}'(z_{1,i})
$$

如果某个 $z_{1,i}\le 0$，那么 $\operatorname{ReLU}'(z_{1,i})=0$，这个位置的梯度就会被截断；如果 $z_{1,i}>0$，那么 $\operatorname{ReLU}'(z_{1,i})=1$，梯度可以原样传回去。

$$
\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1}x^T
$$

$$
\frac{\partial L}{\partial b_1}=\frac{\partial L}{\partial z_1}
$$



## 最终公式汇总

把上面的推导合在一起，单个样本、列向量约定下：

$$
z_1=W_1x+b_1
$$

$$
h=\operatorname{ReLU}(z_1)
$$

$$
\hat y=W_2h+b_2
$$

先记上游梯度为：

$$
g=\frac{\partial L}{\partial \hat y}
$$

那么输出层参数梯度是：

$$
\frac{\partial L}{\partial W_2}=g h^T
$$

$$
\frac{\partial L}{\partial b_2}=g
$$

隐藏层先把梯度传回 $z_1$：

$$
\frac{\partial L}{\partial z_1}
=
\left(W_2^Tg\right)\odot \operatorname{ReLU}'(z_1)
$$

然后隐藏层参数梯度是：

$$
\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1}x^T
$$

$$
\frac{\partial L}{\partial b_1}=\frac{\partial L}{\partial z_1}
$$

这四个参数梯度可以记成一条反向传播链：

$$
\boxed{\frac{\partial L}{\partial W_2}=g h^T}
\quad
\boxed{\frac{\partial L}{\partial b_2}=g}
$$

$$
\boxed{\frac{\partial L}{\partial W_1}=\left((W_2^Tg)\odot \operatorname{ReLU}'(z_1)\right)x^T}
\quad
\boxed{\frac{\partial L}{\partial b_1}=\left(W_2^Tg\right)\odot \operatorname{ReLU}'(z_1)}
$$

上面这些公式是列向量约定：

$$
z_1=W_1x+b_1,
\quad
\hat y=W_2h+b_2
$$

代码里采用 PyTorch / NumPy 常见的 batch 行向量写法：

$$
z_{1,code}=XW_{1,code}+b_1,
\quad
\hat Y_{code}=HW_{2,code}+b_2
$$

所以代码里的权重不是数学公式里的同一个矩阵，而是它的转置版本：

$$
W_{1,code}=W_1^T,
\quad
W_{2,code}=W_2^T
$$

因此参数梯度也要跟着转置：

$$
\left(\frac{\partial L}{\partial W_2}\right)_{code}
=
\left(\frac{\partial L}{\partial W_2}\right)^T
=
\left(g h^T\right)^T
=
h g^T
$$

单个样本如果写成行向量，就是：

$$
dW_{2,code}=h_{row}^T g_{row}
$$

对应 batch 时，把所有样本的贡献加起来：

$$
dW_{2,code}=H^T dY
$$

同理：

$$
dh_{code}=dY W_{2,code}^T
$$

$$
dW_{1,code}=X^T dZ_1
$$

也就是说，代码里出现的 `h.T @ dy`、`dy @ W2.T`、`X.T @ dz1`，对的是“行向量代码约定下的参数形状”，不是把列向量公式里的每一项机械地原样搬过去。

对应到 PyTorch 常见的 batch 行向量写法，设：

| 变量 | PyTorch 形状 |
|---|---:|
| `X` | `(N, D)` |
| `W1` | `(D, I)` |
| `b1` | `(I,)` |
| `z1`, `h` | `(N, I)` |
| `W2` | `(I, J)` |
| `b2` | `(J,)` |
| `y_hat`, `Y` | `(N, J)` |

手写反向传播可以写成：

```python
import torch

z1 = X @ W1 + b1
h = torch.relu(z1)
y_hat = h @ W2 + b2

# 如果 loss = ((y_hat - Y) ** 2).mean() / 2，
# 上游梯度 dy 就要除以元素个数。
dy = (y_hat - Y) / y_hat.numel()

dW2 = h.T @ dy
db2 = dy.sum(dim=0)

dh = dy @ W2.T
dz1 = dh * (z1 > 0).to(z1.dtype)

dW1 = X.T @ dz1
db1 = dz1.sum(dim=0)
```

如果使用 PyTorch autograd，上面这些梯度会自动计算到 `.grad` 里：

```python
W1 = torch.randn(D, I, requires_grad=True)
b1 = torch.zeros(I, requires_grad=True)
W2 = torch.randn(I, J, requires_grad=True)
b2 = torch.zeros(J, requires_grad=True)

z1 = X @ W1 + b1
h = torch.relu(z1)
y_hat = h @ W2 + b2
loss = ((y_hat - Y) ** 2).mean() / 2

loss.backward()

# 对应上面的四个公式
W1.grad  # dL/dW1
b1.grad  # dL/db1
W2.grad  # dL/dW2
b2.grad  # dL/db2
```



## 列向量公式和行向量代码对照

最后把两种写法放在一起看。核心区别只有一个：数学推导默认单个样本是列向量，代码里为了方便组织 batch，默认每个样本是一行。

| 对比项 | 数学推导：列向量 | PyTorch / NumPy：batch 行向量 |
|---|---|---|
| 单个样本输入 | $x\in\mathbb{R}^{D\times 1}$ | `x_row` 的形状是 `(1, D)` |
| 一批样本输入 | $X=[x_1,x_2,\cdots,x_N]\in\mathbb{R}^{D\times N}$ | `X` 的形状是 `(N, D)` |
| 第一层权重 | $W_1\in\mathbb{R}^{I\times D}$ | `W1` 的形状是 `(D, I)`，等于数学里的 $W_1^T$ |
| 第一层前向 | $z_1=W_1x+b_1$ | `z1 = X @ W1 + b1` |
| 隐藏层激活 | $h=\operatorname{ReLU}(z_1)$ | `h = torch.relu(z1)` |
| 第二层权重 | $W_2\in\mathbb{R}^{J\times I}$ | `W2` 的形状是 `(I, J)`，等于数学里的 $W_2^T$ |
| 第二层前向 | $\hat y=W_2h+b_2$ | `y_hat = h @ W2 + b2` |
| 输出层上游梯度 | $g=\frac{\partial L}{\partial \hat y}\in\mathbb{R}^{J\times 1}$ | `dy` 的形状是 `(N, J)` |
| 第二层权重梯度 | $\frac{\partial L}{\partial W_2}=g h^T$ | `dW2 = h.T @ dy`，形状 `(I, J)`，等于数学梯度的转置 |
| 第二层偏置梯度 | $\frac{\partial L}{\partial b_2}=g$ | `db2 = dy.sum(dim=0)` |
| 传回隐藏层 | $\frac{\partial L}{\partial h}=W_2^Tg$ | `dh = dy @ W2.T` |
| ReLU 反向 | $\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial h}\odot\operatorname{ReLU}'(z_1)$ | `dz1 = dh * (z1 > 0).to(z1.dtype)` |
| 第一层权重梯度 | $\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1}x^T$ | `dW1 = X.T @ dz1`，形状 `(D, I)`，等于数学梯度的转置 |
| 第一层偏置梯度 | $\frac{\partial L}{\partial b_1}=\frac{\partial L}{\partial z_1}$ | `db1 = dz1.sum(dim=0)` |

可以把它记成一句话：

> 数学推导里权重左乘列向量；代码实现里样本行向量右乘权重。因此代码里的权重矩阵和权重梯度，都是数学列向量约定下对应矩阵的转置版本。
